In [ ]:
"""
                  Echo Client
                      │
                      │ GET Agent Card
                      ▼
              A2ACardResolver
                      │
                      ▼
                 AgentCard
                      │
                      │ create_client()
                      ▼
                  A2A Client
                      │
                      │ message/send
                      ▼
              SendMessageRequest
                      │
                      ▼
                   Message
                      │
                      ▼
                    Part
                      │
                      │ "Hello Echo Agent!"
                      ▼
                 Echo Agent
                      │
                      ▼
              async for event
                      │
             ┌────────┴────────┐
             ▼                 ▼
         message             task
         artifact       status_update
"""

In [2]:
# ============================================================
# A2A ECHO CLIENT  (a2a-sdk 1.x)
# ============================================================
#
# FLOW
#
#   httpx.AsyncClient
#          |
#          v
#   A2ACardResolver ---GET /.well-known/agent-card.json---> Echo Agent
#          |
#          v
#   AgentCard  (protobuf message)
#          |
#          v
#   await create_client(card, ClientConfig(httpx_client=...))
#          |
#          v
#   SendMessageRequest( Message( role=ROLE_USER, parts=[Part(text=...)] ) )
#          |
#          v
#   async for event in client.send_message(request)  ---> Echo Agent
#          |
#          v
#   StreamResponse  (one of: message | task | status_update | artifact_update)
#
# WHY THE OLD CODE FAILED
#
#   In a2a-sdk 1.x all A2A types (AgentCard, Message, Part, ...)
#   are protobuf messages, not Pydantic models.
#   Protobuf messages have no .model_dump().
#   Use google.protobuf.json_format.MessageToJson / MessageToDict.
#
# WHAT THIS DOES NOT DO
#
#   - It does not start the Echo Agent. The server must already
#     be running on AGENT_URL.
#   - It does not handle auth. No interceptors are configured.
# ============================================================

import uuid

import httpx
from google.protobuf.json_format import MessageToJson

from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Message, Part, Role, SendMessageRequest


AGENT_URL = "http://localhost:9111"


def print_proto(title, proto_msg):
    # Protobuf -> JSON. This replaces model_dump(mode="json").
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    print(MessageToJson(proto_msg, indent=2))


async def main():

    print("=" * 60)
    print("A2A ECHO CLIENT")
    print("=" * 60)

    async with httpx.AsyncClient() as httpx_client:

        # ----------------------------------------------------
        # STEP 1: Fetch the Agent Card
        # ----------------------------------------------------
        # GET {AGENT_URL}/.well-known/agent-card.json
        resolver = A2ACardResolver(httpx_client, AGENT_URL)
        agent_card = await resolver.get_agent_card()

        print(agent_card)
        print("\nAgent:", agent_card.name)
        print_proto("AGENT CARD", agent_card)

        # ----------------------------------------------------
        # STEP 2: Create the A2A client
        # ----------------------------------------------------
        # create_client is async in 1.x -> must be awaited.
        # httpx_client goes inside ClientConfig, not as a kwarg.
        # streaming=False -> one blocking message/send call.
        # (With True, SDK streams only if the card says the
        #  agent supports streaming.)
        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=agent_card.capabilities.streaming,
        )
        client = await create_client(agent_card, client_config=config)
        #
        # ----------------------------------------------------
        # STEP 3: Build the request
        # ----------------------------------------------------
        # send_message does NOT accept a plain string.
        # It needs SendMessageRequest -> Message -> Part.
        request = SendMessageRequest(
            message=Message(
                message_id=str(uuid.uuid4()),
                role=Role.ROLE_USER,
                parts=[Part(text="Hello Echo Agent!")],
            )
        )

        # # ----------------------------------------------------
        # # STEP 4: Send and read responses
        # # ----------------------------------------------------
        # # send_message returns an async iterator, not a value.
        # # Non-streaming mode yields exactly one StreamResponse.
        # print("\nSending message...")
        #
        async for event in client.send_message(request):

            kind = event.WhichOneof("payload")

            if kind == "message":
                # Direct reply: print the text parts.
                texts = [p.text for p in event.message.parts if p.text]
                print("\nReply:", " ".join(texts))

            print_proto(f"RAW RESPONSE ({kind})", event)


# ============================================================
# RUNNING
# ============================================================
# Jupyter: the cell already has an event loop.
#     await main()
#
# Plain script: replace the line below with
#     import asyncio; asyncio.run(main())
# ============================================================

await main()

A2A ECHO CLIENT
name: "echo"
description: "Reverses text. Exists to demonstrate the A2A protocol."
supported_interfaces {
  url: "http://127.0.0.1:9111/"
  protocol_binding: "JSONRPC"
  protocol_version: "1.0"
}
version: "1.0.0"
capabilities {
  streaming: false
}
default_input_modes: "text/plain"
default_output_modes: "text/plain"
skills {
  id: "reverse"
  name: "Reverse text"
  description: "Returns the input text backwards."
  tags: "a2a"
  tags: "learning-example"
  examples: "hello world"
  input_modes: "text/plain"
  output_modes: "text/plain"
}


Agent: echo

AGENT CARD
{
  "name": "echo",
  "description": "Reverses text. Exists to demonstrate the A2A protocol.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:9111/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "sk